In [0]:
VOLUME_PATH = "/Volumes/workspace/retail_sales_dw/file_folder"

dbutils.fs.rm(VOLUME_PATH, True)

dbutils.fs.mkdirs(VOLUME_PATH)

print(f"Cleared: {VOLUME_PATH}")

In [0]:
# Databricks notebook source

# ============================================================
# 1. IMPORT
# ============================================================

from datetime import datetime, timedelta
import random
import csv
import os


# ============================================================
# 2. CONFIGURATION
# ============================================================

# กำหนดช่วงวันที่ที่ต้องการ Generate
START_DATE = "2026-01-01"
END_DATE   = "2026-01-05"

# จำนวน rows ต่อวัน
ROWS_PER_DAY = 200

# จำนวน unique keys ที่ถูกต้องต่อวัน
UNIQUE_KEYS_PER_DAY = 150

# Databricks Volume
# OUTPUT_DIR = "/Volumes/workspace/retail_sales_dw/file_folder"
OUTPUT_DIR = "/Workspace/Users/worada.wongtayan@gmail.com/Retail-Sales-ETL-Pipeline/file_to_volume"


# Seed เพื่อให้สามารถ reproduce data ได้
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


# ============================================================
# 3. MASTER DATA
# ============================================================

SHOP_NAMES = [
    "James",
    "John",
    "Michael",
    "William",
    "David",
    "Robert",
    "Daniel",
    "Christopher",
    "Matthew",
    "Andrew",
    "Joseph",
    "Thomas",
    "Charles",
    "Benjamin",
    "Alexander",
    "Nicholas",
    "Ethan",
    "Ryan",
    "Jacob"
]

BRANCH_NAMES = [
    "Bangkok",
    "Chiang Mai",
    "Phuket",
    "Pattaya",
    "Khon Kaen",
    "Hat Yai",
    "Nakhon Ratchasima",
    "Udon Thani",
    "Ayutthaya",
    "Chonburi"
]


# ============================================================
# 4. DATA QUALITY ERROR CONFIGURATION
# ============================================================

# จำนวนข้อมูลผิดแต่ละประเภทต่อวัน
ERROR_CONFIG = {
    "shop_id_null": 5,
    "shop_id_string": 5,
    "file_dt_string": 10,
    "row_duplicate": 5,
    "key_duplicate": 10,
}


# ============================================================
# 5. DATE RANGE
# ============================================================

def get_date_range(start_date: str, end_date: str):

    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date()

    if start > end:
        raise ValueError(
            f"START_DATE {start_date} must be <= END_DATE {end_date}"
        )

    current = start

    while current <= end:
        yield current
        current += timedelta(days=1)


# ============================================================
# 6. GENERATE NORMAL ROW
# ============================================================

def generate_normal_row(shop_id, file_dt):

    return {
        "shop_id": shop_id,
        "shop_name": random.choice(SHOP_NAMES),
        "branch_name": random.choice(BRANCH_NAMES),
        "file_dt": file_dt
    }


# ============================================================
# 7. GENERATE DAILY DATA
# ============================================================

def generate_daily_data(file_dt):

    rows = []

    # --------------------------------------------------------
    # สร้าง 150 unique keys
    # --------------------------------------------------------

    # key ถูกออกแบบให้ไม่ซ้ำกันในแต่ละวัน
    #
    # ตัวอย่าง:
    # 20260401 -> 20260401001 ... 20260401150
    #
    # date_key = file_dt.strftime("%Y%m%d")

    unique_shop_ids = [
        int(f"{i:05d}")
        for i in range(1, UNIQUE_KEYS_PER_DAY + 1)
    ]

    # --------------------------------------------------------
    # 150 rows แรก = unique keys
    # --------------------------------------------------------

    for shop_id in unique_shop_ids:

        row = generate_normal_row(
            shop_id=shop_id,
            file_dt=file_dt
        )

        rows.append(row)

    # --------------------------------------------------------
    # อีก 50 rows
    #
    # ทำให้เกิด duplicate keys
    # --------------------------------------------------------

    duplicate_key_count = ROWS_PER_DAY - UNIQUE_KEYS_PER_DAY

    for _ in range(duplicate_key_count):

        shop_id = random.choice(unique_shop_ids)

        row = generate_normal_row(
            shop_id=shop_id,
            file_dt=file_dt
        )

        rows.append(row)

    # --------------------------------------------------------
    # Shuffle
    # --------------------------------------------------------

    random.shuffle(rows)

    return rows


# ============================================================
# 8. INJECT DATA QUALITY ERRORS
# ============================================================

def inject_data_quality_errors(rows, file_dt):

    total_rows = len(rows)

    # --------------------------------------------------------
    # 8.1 shop_id = NULL
    # --------------------------------------------------------

    count = ERROR_CONFIG["shop_id_null"]

    indexes = random.sample(
        range(total_rows),
        count
    )

    for idx in indexes:

        rows[idx]["shop_id"] = None


    # --------------------------------------------------------
    # 8.2 shop_id = STRING
    # --------------------------------------------------------

    count = ERROR_CONFIG["shop_id_string"]

    available_indexes = [
        i for i in range(total_rows)
        if rows[i]["shop_id"] is not None
    ]

    indexes = random.sample(
        available_indexes,
        count
    )

    for idx in indexes:

        # เปลี่ยน int เป็น string
        rows[idx]["shop_id"] = str(rows[idx]["shop_id"])


    # --------------------------------------------------------
    # 8.3 file_dt = STRING
    # --------------------------------------------------------

    count = ERROR_CONFIG["file_dt_string"]

    indexes = random.sample(
        range(total_rows),
        count
    )

    for idx in indexes:

        rows[idx]["file_dt"] = file_dt.strftime("%Y-%m-%d")


    # --------------------------------------------------------
    # 8.4 ROW DUPLICATE
    #
    # duplicate ทั้ง row
    # --------------------------------------------------------

    count = ERROR_CONFIG["row_duplicate"]

    for _ in range(count):

        source_row = random.choice(rows)

        duplicate_row = source_row.copy()

        rows.append(duplicate_row)


    # --------------------------------------------------------
    # 8.5 KEY DUPLICATE
    #
    # key ซ้ำ แต่ข้อมูล column อื่นอาจแตกต่าง
    # --------------------------------------------------------

    count = ERROR_CONFIG["key_duplicate"]

    for _ in range(count):

        source_row = random.choice(rows)

        duplicate_key_row = {
            "shop_id": source_row["shop_id"],
            "shop_name": random.choice(SHOP_NAMES),
            "branch_name": random.choice(BRANCH_NAMES),
            "file_dt": source_row["file_dt"]
        }

        rows.append(duplicate_key_row)


    return rows


# ============================================================
# 9. NORMALIZE ROW COUNT
# ============================================================

def normalize_row_count(rows):

    # หลัง inject errors จำนวน rows จะมากกว่า 200
    #
    # ตัดกลับมาเป็น 200 rows
    #
    # เพื่อให้ทุกวันมี exactly 200 rows

    if len(rows) > ROWS_PER_DAY:

        rows = rows[:ROWS_PER_DAY]

    elif len(rows) < ROWS_PER_DAY:

        while len(rows) < ROWS_PER_DAY:

            row = random.choice(rows).copy()

            rows.append(row)

    return rows


# ============================================================
# 10. WRITE CSV
# ============================================================

def write_csv(rows, file_dt):

    file_name = f"shop_name_{file_dt.strftime('%Y%m%d')}.csv"

    output_path = os.path.join(
        OUTPUT_DIR,
        file_name
    )

    fieldnames = [
        "shop_id",
        "shop_name",
        "branch_name",
        "file_dt"
    ]

    with open(
        output_path,
        mode="w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames
        )

        writer.writeheader()

        writer.writerows(rows)

    return output_path


# ============================================================
# 11. VALIDATION
# ============================================================

def validate_daily_data(rows, file_dt):

    print("=" * 70)

    print(
        f"DATE        : {file_dt}"
    )

    print(
        f"TOTAL ROWS  : {len(rows)}"
    )

    # --------------------------------------------------------
    # unique keys
    # --------------------------------------------------------

    valid_shop_ids = [
        row["shop_id"]
        for row in rows
        if row["shop_id"] is not None
    ]

    unique_keys = len(set(valid_shop_ids))

    print(
        f"UNIQUE KEYS  : {unique_keys}"
    )

    # --------------------------------------------------------
    # NULL shop_id
    # --------------------------------------------------------

    null_shop_id = sum(
        row["shop_id"] is None
        for row in rows
    )

    print(
        f"NULL shop_id : {null_shop_id}"
    )

    # --------------------------------------------------------
    # STRING shop_id
    # --------------------------------------------------------

    string_shop_id = sum(
        isinstance(row["shop_id"], str)
        for row in rows
    )

    print(
        f"STRING shop_id : {string_shop_id}"
    )

    # --------------------------------------------------------
    # file_dt type
    # --------------------------------------------------------

    string_file_dt = sum(
        isinstance(row["file_dt"], str)
        for row in rows
    )

    print(
        f"STRING file_dt : {string_file_dt}"
    )

    print("=" * 70)


# ============================================================
# 12. MAIN
# ============================================================

def main():

    print("=" * 70)
    print("RETAIL SALES DATA GENERATOR")
    print("=" * 70)

    print(
        f"START DATE          : {START_DATE}"
    )

    print(
        f"END DATE            : {END_DATE}"
    )

    print(
        f"ROWS PER DAY        : {ROWS_PER_DAY}"
    )

    print(
        f"UNIQUE KEYS PER DAY : {UNIQUE_KEYS_PER_DAY}"
    )

    print(
        f"OUTPUT DIRECTORY     : {OUTPUT_DIR}"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # ตรวจสอบ output directory
    # --------------------------------------------------------

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Generate ทุกวัน
    # --------------------------------------------------------

    total_days = 0
    total_rows = 0

    for file_dt in get_date_range(
        START_DATE,
        END_DATE
    ):

        print(
            f"\nGenerating data for {file_dt} ..."
        )

        # 1. generate normal data
        rows = generate_daily_data(
            file_dt
        )

        # 2. inject bad data
        rows = inject_data_quality_errors(
            rows,
            file_dt
        )

        # 3. normalize กลับมา 200 rows
        rows = normalize_row_count(
            rows
        )

        # 4. validation
        validate_daily_data(
            rows,
            file_dt
        )

        # 5. write CSV
        output_path = write_csv(
            rows,
            file_dt
        )

        print(
            f"Created: {output_path}"
        )

        total_days += 1
        total_rows += len(rows)


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print("\n")
    print("=" * 70)
    print("GENERATION COMPLETED")
    print("=" * 70)

    print(
        f"TOTAL DAYS : {total_days}"
    )

    print(
        f"TOTAL ROWS : {total_rows}"
    )

    print(
        f"OUTPUT     : {OUTPUT_DIR}"
    )

    print("=" * 70)


# ============================================================
# RUN
# ============================================================

main()

In [0]:
import random
import pandas as pd

from datetime import datetime, timedelta


# ============================================================
# CONFIG
# ============================================================

ROWS_PER_DAY = 200

SEED = 42

random.seed(SEED)


# ============================================================
# GENERATE FACT SALES
# ============================================================

def generate_fact_sales(
    start_date: str,
    end_date: str,
    rows_per_day: int = ROWS_PER_DAY
) -> pd.DataFrame:

    """
    Generate fact_sales data.

    Columns:
        sales_date
        transaction_id
        shop_id
        sales_qty
        sales_amt

    Data Quality test cases:

        5  Invalid records
        5  Null-key records
        5  Key-duplicate records
        5  Row-duplicate records

    Expected result:

        Bronze       = 200
        Invalid      = 5
        Null key     = 5
        Duplicate    = 10
        All bad      = 20
        Final        = 180
    """

    start = datetime.strptime(
        start_date,
        "%Y-%m-%d"
    ).date()

    end = datetime.strptime(
        end_date,
        "%Y-%m-%d"
    ).date()

    all_records = []

    current_date = start

    # ========================================================
    # LOOP BY DATE
    # ========================================================

    while current_date <= end:

        records = []

        # ====================================================
        # 1. GENERATE NORMAL DATA
        # ====================================================

        for i in range(rows_per_day):

            transaction_id = int(
                current_date.strftime("%Y%m%d")
                + f"{i + 1:03d}"
            )

            shop_id = random.randint(
                1,
                150
            )

            sales_qty = random.randint(
                1,
                20
            )

            sales_amt = round(
                random.uniform(
                    100,
                    10000
                ),
                2
            )

            records.append({

                "sales_date":
                    current_date,

                "transaction_id":
                    transaction_id,

                "shop_id":
                    shop_id,

                "sales_qty":
                    sales_qty,

                "sales_amt":
                    sales_amt
            })

        # ====================================================
        # 2. INVALID RECORD = 5
        # ====================================================
        #
        # sales_qty ต้องเป็น INT
        #
        # แต่ใส่ STRING ที่ cast เป็น INT ไม่ได้
        #
        # rows: 0-4
        # ====================================================

        for i in range(0, 5):

            records[i]["sales_qty"] = "INVALID"


        # ====================================================
        # 3. NULL KEY = 5
        # ====================================================
        #
        # keys:
        #   transaction_id
        #   shop_id
        #
        # ทำ transaction_id เป็น NULL
        #
        # rows: 5-9
        # ====================================================

        for i in range(5, 10):

            records[i]["transaction_id"] = None


        # ====================================================
        # 4. KEY DUPLICATE = 5
        # ====================================================
        #
        # rows 10-14
        #
        # duplicate key กับ rows 20-24
        #
        # key:
        #   transaction_id
        #   shop_id
        #
        # แต่ sales_qty แตกต่างกัน
        #
        # ดังนั้น:
        #
        #   key เหมือนกัน
        #   row ไม่เหมือนกัน
        #
        # => _key_duplicate
        # ====================================================

        for i in range(10, 15):

            source = i + 10

            records[i]["transaction_id"] = (
                records[source]["transaction_id"]
            )

            records[i]["shop_id"] = (
                records[source]["shop_id"]
            )

            records[i]["sales_qty"] = (
                records[source]["sales_qty"] + 1
            )


        # ====================================================
        # 5. ROW DUPLICATE = 5
        # ====================================================
        #
        # rows 15-19
        #
        # copy rows 25-29 แบบเหมือนกันทุก column
        #
        # แต่ _sk จะถูกสร้างใน Bronze/Silver ภายหลัง
        #
        # ดังนั้น:
        #
        #   data columns เหมือนกัน
        #   _sk ต่างกัน
        #
        # => _row_duplicate
        # ====================================================

        for i in range(15, 20):

            source = i + 10

            records[i] = records[source].copy()


        # ====================================================
        # 6. APPEND TO ALL RECORDS
        # ====================================================

        all_records.extend(records)

        current_date += timedelta(days=1)


    # ========================================================
    # CREATE DATAFRAME
    # ========================================================

    df = pd.DataFrame(
        all_records,
        columns=[
            "sales_date",
            "transaction_id",
            "shop_id",
            "sales_qty",
            "sales_amt"
        ]
    )


    # ========================================================
    # RETURN
    # ========================================================

    return df


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    df = generate_fact_sales(
        start_date="2026-01-01",
        end_date="2026-01-01",
        rows_per_day=200
    )

    print(
        f"Generated rows: {len(df)}"
    )

    print("\nSchema:")
    print(
        df.dtypes
    )

    print("\nSample:")
    print(
        df.head(20)
    )